In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/clean_retail.csv",
                 parse_dates=["InvoiceDate"])

# Cut-off: aakhri tareekh se 90 din pehle
last_date = df["InvoiceDate"].max()
cutoff = last_date - pd.Timedelta(days=90)
print("Aakhri tareekh:", last_date)
print("Cut-off:", cutoff)

past = df[df["InvoiceDate"] <= cutoff]     # features isi se banenge
future = df[df["InvoiceDate"] > cutoff]    # label isi se banega
print("Past rows:", len(past), "| Future rows:", len(future))

# Cut-off par RFM (sirf past data se)
snapshot = cutoff + pd.Timedelta(days=1)
feat = past.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("TotalPrice", "sum"),
).reset_index()

# Label: future me kuch khareeda? Nahi khareeda to churn = 1
buyers_future = future["Customer ID"].unique()
feat["Churn"] = (~feat["Customer ID"].isin(buyers_future)).astype(int)

print("Customers:", feat.shape[0])
print(feat["Churn"].value_counts(normalize=True).round(3))

Aakhri tareekh: 2011-12-09 12:50:00
Cut-off: 2011-09-10 12:50:00
Past rows: 619306 | Future rows: 158606
Customers: 5245
Churn
1    0.565
0    0.435
Name: proportion, dtype: float64


In [ ]:
feat.to_csv("../data/processed/churn_features.csv", index=False)

In [ ]:
# Extra features (sirf past data se, taake leakage na ho)
extra = past.groupby("Customer ID").agg(
    FirstPurchase=("InvoiceDate", "min"),
    UniqueProducts=("StockCode", "nunique"),
).reset_index()
extra["Tenure"] = (snapshot - extra["FirstPurchase"]).dt.days

recent = past[past["InvoiceDate"] > cutoff - pd.Timedelta(days=90)]
orders_90 = (recent.groupby("Customer ID")["Invoice"].nunique()
             .rename("Orders_Last90").reset_index())

feat2 = feat.merge(extra[["Customer ID", "Tenure", "UniqueProducts"]],
                   on="Customer ID", how="left")
feat2 = feat2.merge(orders_90, on="Customer ID", how="left")
feat2["Orders_Last90"] = feat2["Orders_Last90"].fillna(0)
feat2["AvgOrderValue"] = feat2["Monetary"] / feat2["Frequency"]

feat2.to_csv("../data/processed/churn_features_v2.csv", index=False)
print(feat2.shape)
feat2.head()

(5245, 9)


,Customer ID,Recency,Frequency,Monetary,Churn,Tenure,UniqueProducts,Orders_Last90,AvgOrderValue
0,12346,439,11,372.86,1,636,26,0.0,33.896364
1,12347,40,6,3402.39,0,314,107,1.0,567.065000
2,12348,159,4,1709.40,0,348,25,0.0,427.350000
3,12349,318,2,2221.14,0,499,89,0.0,1110.570000
4,12350,220,1,334.40,1,220,17,0.0,334.400000


Goal: churn label. Cut-off 10 Sep 2011, features from before it, label from the 90 days after. 5,245 customers, 56.5% churned. 4 extra features added.

x